# ModernFinBERT v2 Stage 1: Short-Context Training

Train ModernBERT-base with LoRA on short-context financial data (headlines, tweets, analyst reports, earnings excerpts).
- Dataset: `neoyipeng/modernfinbert-training-v2` (34.6K train rows)
- Entity-aware input: `[CLS] entity [SEP] text [SEP]` — trains entity-targeted sentiment
- Training target: `entity_sentiment` (NEGATIVE/NEUTRAL/POSITIVE)
- Pushes merged model to HuggingFace for Stage 2 (long-context fine-tuning in NB 01b)

Estimated time on T4: ~10-15 minutes

In [ ]:
%%capture
import os, re, torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer scikit-learn
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
%env UNSLOTH_DISABLE_FAST_GENERATION=1

import torch
import numpy as np
from unsloth import FastModel, is_bfloat16_supported
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 3407
NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
LABEL_TO_ID = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
ID_TO_LABEL = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
MAX_LENGTH = 512
MODEL_NAME = "unsloth/ModernBERT-base"
HF_PUSH_NAME = "neoyipeng/ModernFinBERT-v2-short"

cap = torch.cuda.get_device_capability()
assert cap[0] >= 7, f"GPU sm_{cap[0]}{cap[1]} not supported — need sm_70+ (T4/V100/A100)"
print(f"GPU: {torch.cuda.get_device_name()} (sm_{cap[0]}{cap[1]})")
print(f"Max tokens: {MAX_LENGTH}")

## 1. Load Short-Context Data

In [ ]:
ds = load_dataset("neoyipeng/modernfinbert-training-v2")
print(ds)

# Keep only needed columns
for split in ds:
    ds[split] = ds[split].select_columns(["text", "entity", "entity_sentiment"])

# Map entity_sentiment to integer labels
def map_labels(examples):
    examples["label"] = [LABEL_TO_ID[l] for l in examples["entity_sentiment"]]
    return examples

for split in ds:
    ds[split] = ds[split].map(map_labels, batched=True)
    ds[split] = ds[split].remove_columns(["entity_sentiment"])

print(f"\nLabel distribution (train, entity_sentiment):")
labels = ds["train"]["label"]
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name}: {labels.count(i)}")

entities = ds["train"]["entity"]
n_with_entity = sum(1 for e in entities if e not in ("NONE", "MARKET", "None", "", None))
print(f"\nEntity coverage: {n_with_entity}/{len(entities)} ({100*n_with_entity/len(entities):.1f}%)")

## 2. Model + LoRA Setup

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    auto_model=AutoModelForSequenceClassification,
    max_seq_length=MAX_LENGTH,
    dtype=None,
    num_labels=NUM_CLASSES,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    load_in_4bit=False,
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    task_type="SEQ_CLS",
)

In [ ]:
# Entity-aware tokenization: sentence pair format [CLS] entity [SEP] text [SEP]
# For rows without a meaningful entity, pass empty string as sentence A.
def tokenize_function(examples):
    entities = [
        e if e not in ("NONE", "MARKET", "None", "", None) else ""
        for e in examples["entity"]
    ]
    return tokenizer(entities, examples["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = {}
for split in ds:
    tokenized[split] = ds[split].map(tokenize_function, batched=True, remove_columns=["text", "entity"])
    tokenized[split] = tokenized[split].rename_column("label", "labels")
    print(f"{split}: {len(tokenized[split])} rows tokenized")

## 3. Training

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

# Short data: median ~44 tokens, batch=32 fits easily on T4.
# 3 epochs, effective batch = 32 * 2 = 64.
# Estimated: ~541 steps/epoch * 3 = ~1,623 steps, ~10-15 min on T4.

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    args=TrainingArguments(
        output_dir="./modernfinbert-v2-short-output",
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        weight_decay=0.001,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=SEED,
        group_by_length=True,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
    ),
)

trainer.train()

In [ ]:
test_results = trainer.evaluate(tokenized["test"])
print(f"Test: accuracy={test_results['eval_accuracy']:.4f}, macro_f1={test_results['eval_macro_f1']:.4f}")

# Per-class breakdown
preds = trainer.predict(tokenized["test"])
y_pred = preds.predictions.argmax(axis=-1)
y_true = preds.label_ids
print(f"\n{classification_report(y_true, y_pred, target_names=LABEL_NAMES)}")

print(f"\nTraining log:")
for entry in trainer.state.log_history:
    if "eval_loss" in entry:
        print(f"  Epoch {entry.get('epoch', '?')}: loss={entry['eval_loss']:.4f}, "
              f"acc={entry.get('eval_accuracy', 0):.4f}, f1={entry.get('eval_macro_f1', 0):.4f}")

## 4. Push Stage-1 Model to HuggingFace

Merge LoRA weights and push. This model is loaded by NB 01b for long-context fine-tuning.

In [ ]:
merged = model.merge_and_unload()
merged.push_to_hub(HF_PUSH_NAME, private=False)
tokenizer.push_to_hub(HF_PUSH_NAME, private=False)
print(f"Stage-1 model pushed to {HF_PUSH_NAME}")
print(f"Next: run NB 01b to fine-tune on long-context data.")